<a href="https://colab.research.google.com/github/YassGan/3DGaussianSplatting-INRIA-Method-Colab/blob/feat%2Fworking_with_drive_ds/working_with_drive_ds/Copy_of_3DGaussianSplatting_INRIA_Method_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Making sure that we are using a GPU

In [1]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


# Installing pycolmap before any other installation or modificattion

In [ ]:
!pip install pycolmap

# Python downgrading


In [ ]:
!wget -O mini.sh https://repo.anaconda.com/miniconda/Miniconda3-py37_23.1.0-1-Linux-x86_64.sh
!chmod +x mini.sh
!bash ./mini.sh -b -f -p /usr/local
!conda install -q -y python=3.7
import sys
sys.path.append('/usr/local/lib/python3.7/site-packages')
!python --version  # Should say Python 3.7.x

# CUDA 11.8

In [ ]:
!wget https://developer.download.nvidia.com/compute/cuda/11.8.0/local_installers/cuda_11.8.0_520.61.05_linux.run
!chmod +x cuda_11.8.0_520.61.05_linux.run
!./cuda_11.8.0_520.61.05_linux.run --silent --toolkit --no-drm --no-man-page
import os
os.environ['PATH'] += ':/usr/local/cuda-11.8/bin'
os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda-11.8/lib64:/usr/lib64-nvidia'
!nvcc --version  # Should show CUDA 11.8

--2025-03-05 02:03:52--  https://developer.download.nvidia.com/compute/cuda/11.8.0/local_installers/cuda_11.8.0_520.61.05_linux.run
Resolving developer.download.nvidia.com (developer.download.nvidia.com)... 104.123.70.41, 104.123.70.66, 104.123.70.65, ...
Connecting to developer.download.nvidia.com (developer.download.nvidia.com)|104.123.70.41|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4336730777 (4.0G) [application/octet-stream]
Saving to: ‘cuda_11.8.0_520.61.05_linux.run’

cuda_11.8.0_520.61. 100%[===================>]   4.04G   107MB/s    in 45s     

2025-03-05 02:04:38 (91.9 MB/s) - ‘cuda_11.8.0_520.61.05_linux.run’ saved [4336730777/4336730777]

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2022 NVIDIA Corporation
Built on Wed_Sep_21_10:33:58_PDT_2022
Cuda compilation tools, release 11.8, V11.8.89
Build cuda_11.8.r11.8/compiler.31833905_0


#Pytorch with cuda

In [ ]:
!pip uninstall torch torchvision torchaudio -y
!pip install torch==1.12.1+cu116 torchvision==0.13.1+cu116 torchaudio==0.12.1 --extra-index-url https://download.pytorch.org/whl/cu116

Found existing installation: torch 2.5.1+cu124
Uninstalling torch-2.5.1+cu124:
  Successfully uninstalled torch-2.5.1+cu124
Found existing installation: torchvision 0.20.1+cu124
Uninstalling torchvision-0.20.1+cu124:
  Successfully uninstalled torchvision-0.20.1+cu124
Found existing installation: torchaudio 2.5.1+cu124
Uninstalling torchaudio-2.5.1+cu124:
  Successfully uninstalled torchaudio-2.5.1+cu124
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu116
ERROR: Could not find a version that satisfies the requirement torch==1.12.1+cu116 (from versions: 1.13.0, 1.13.0+cu116, 1.13.1, 1.13.1+cu116, 2.0.0, 2.0.1, 2.1.0, 2.1.1, 2.1.2, 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0)
ERROR: No matching distribution found for torch==1.12.1+cu116


# Verification of the versions

In [ ]:
import torch
print(torch.cuda.is_available())  # Should be True
print(torch.version.cuda)        # Should be 11.3 (from PyTorch)
print(torch.cuda.get_device_name(0))  # Should show GPU

ModuleNotFoundError: No module named 'torch'

In [ ]:
##making sure that we are always using the GPU and not a CPU
!nvidia-smi

# Cloning the 3D Gaussian Splatting algorithm and the submodules of the algorithm

In [ ]:
%cd /content
!git clone --recursive https://github.com/camenduru/gaussian-splatting
!pip install -q plyfile

%cd /content/gaussian-splatting
!pip install -q /content/gaussian-splatting/submodules/diff-gaussian-rasterization
!pip install -q /content/gaussian-splatting/submodules/simple-knn


# Mounting the drive

In [ ]:

images_folder_name="Images"

import os
import zipfile
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')
# Paths




# Choosing a specific zip file

In [ ]:

!unzip "/content/drive/MyDrive/Images.zip" -d "/content/dataset"


Working with a file that has the colmap needed files

# COLMAP Code for camera position calculation

In [ ]:
from pathlib import Path
import os
import subprocess
import pycolmap

output_path = Path("output")
image_dir = "images"

output_path.mkdir(exist_ok=True)
mvs_path = output_path / "mvs"
database_path = output_path / "database.db"

pycolmap.extract_features(database_path, image_dir)
pycolmap.match_exhaustive(database_path)
maps = pycolmap.incremental_mapping(database_path, image_dir, output_path)
maps[0].write(output_path)



# Arranging the input folder of the 3D Gaussian splatting method

In [ ]:
import os
import shutil
from pathlib import Path

# Define paths (adjust these to your actual paths)



input_images_folder = "dataset/"+images_folder_name # Folder with your input images
colmap_output_folder = "camera_positions_output"  # Folder containing COLMAP's output (e.g., "sparse" files)
new_parent_folder = "3D_Gaussian_Splatting_input_folder"  # New folder to create

# Create the new parent folder
new_parent = Path(new_parent_folder)
new_parent.mkdir(parents=True, exist_ok=True)

# 1. Copy input images to "images" subfolder
images_subfolder = new_parent / "images"
images_subfolder.mkdir(exist_ok=True)

# Copy all images from input_images_folder to images_subfolder
for img in Path(input_images_folder).glob("*"):
    if img.is_file() and img.suffix.lower() in [".jpg", ".jpeg", ".png"]:
        shutil.copy(img, images_subfolder / img.name)

# 2. Create "sparse" subfolder and copy COLMAP output files
sparse_subfolder = new_parent / "sparse"
sparse_subfolder.mkdir(exist_ok=True)

# Copy COLMAP reconstruction files (cameras.bin, images.bin, points3D.bin)
required_colmap_files = ["cameras.bin", "images.bin", "points3D.bin"]
for file in required_colmap_files:
    src = Path(colmap_output_folder) / file
    if src.exists():
        shutil.copy(src, sparse_subfolder / file)
    else:
        print(f"Warning: {file} not found in COLMAP output folder!")

print(f"Dataset folder created at: {new_parent_folder}")

Training

In [ ]:
# import os
# import zipfile
# from google.colab import drive

# # Mount Google Drive
# drive.mount('/content/drive')
# # Paths
# zip_path = "/content/drive/MyDrive/yass_book.zip"

# !unzip zip_path

# !python train.py -s /content/dataset/yass_book/


!python train.py \
  -s /content/dataset/TestColmap_python \
  -m /content/output \
  --iterations 30000 \
  --test_iterations 1000 2000 3000 4000 \
  --save_iterations 1000 2000 3000 4000
